# GRAFT — Graph Retrieval-Augmented Fine-Tuning (All-in-One)

All code inline — no external `.py` files needed. Just run cells top to bottom.

**Setup:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
!pip install -q torch transformers tqdm pyyaml numpy

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
# Cell 10: Custom Inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForCausalLM.from_pretrained("./graft_model/best").to(device)
model.eval()
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Build a custom scene
scene = SceneGraph(
    objects={"1": SceneObject("1", "car", ["red", "parked"]), "2": SceneObject("2", "tree", ["large"])},
    relations=[SceneRelation("1", "under", "2")],
)
test_sample = GraftSample(
    image_id="custom",
    scene_graph=scene,
    oracle_facts=[KBTriple("tree", "CanDrop", "branches"), KBTriple("falling_branches", "Damage", "vehicle")],
    distractor_facts=[KBTriple("car", "HasColor", "red")],
    query="What risk does the car face?",
)

prompt = build_inference_prompt(test_sample)
enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256, padding=True)
enc = {k: v.to(device) for k, v in enc.items()}

with torch.no_grad():
    gen = model.generate(
        input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
        max_new_tokens=64, do_sample=True, top_k=50, top_p=0.95,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
    )

input_len = enc["input_ids"].shape[1]
answer = tokenizer.decode(gen[0, input_len:], skip_special_tokens=True).strip()
print(f"Query: {test_sample.query}")
print(f"Answer: {answer}")


In [ ]:
# Cell 2: Domain Entities
from dataclasses import dataclass, field
from typing import List, Dict, Optional


@dataclass(frozen=True)
class SceneObject:
    object_id: str
    name: str
    attributes: List[str] = field(default_factory=list)


@dataclass(frozen=True)
class SceneRelation:
    source_id: str
    relation_name: str
    target_id: str


@dataclass
class SceneGraph:
    objects: Dict[str, SceneObject] = field(default_factory=dict)
    relations: List[SceneRelation] = field(default_factory=list)

    def serialize_relations(self) -> str:
        parts = []
        for rel in self.relations:
            src_name = self.objects[rel.source_id].name
            tgt_name = self.objects[rel.target_id].name
            parts.append(f"({src_name} -> {rel.relation_name} -> {tgt_name})")
        return ", ".join(parts)


@dataclass(frozen=True)
class KBTriple:
    e1_label: str
    relation: str
    e2_label: str

    def serialize(self) -> str:
        return f"<{self.e1_label}, {self.relation}, {self.e2_label}>"


@dataclass
class GraftSample:
    image_id: str
    scene_graph: SceneGraph
    oracle_facts: List[KBTriple] = field(default_factory=list)
    distractor_facts: List[KBTriple] = field(default_factory=list)
    query: str = ""
    ground_truth_trajectory: str = ""
    answer: str = ""

    @property
    def all_facts(self) -> List[KBTriple]:
        return self.oracle_facts + self.distractor_facts


@dataclass
class TrainingMetrics:
    epoch: int
    train_loss: float
    val_loss: float
    perplexity: Optional[float] = None


@dataclass
class EvalResult:
    prompt: str
    generated: str
    gold: str
    exact_match: bool
    token_f1: float


print("✅ Domain entities defined")


In [ ]:
# Cell 3: Infrastructure — Sample Loader & KB Retriever

def parse_raw_sample(raw: dict) -> GraftSample:
    """Convert a raw dict sample into a GraftSample entity."""
    sg_data = raw["gqa_scene_graph"]
    objects = {
        obj_id: SceneObject(
            object_id=obj_id,
            name=obj_data["name"],
            attributes=obj_data.get("attributes", []),
        )
        for obj_id, obj_data in sg_data["objects"].items()
    }
    relations = [
        SceneRelation(
            source_id=rel["source"],
            relation_name=rel["name"],
            target_id=rel["target"],
        )
        for rel in sg_data["relations"]
    ]
    scene_graph = SceneGraph(objects=objects, relations=relations)

    oracle_facts = [
        KBTriple(e1_label=f["e1_label"], relation=f["rel"], e2_label=f["e2_label"])
        for f in raw.get("fvqa_graph_rag_facts", [])
    ]
    distractor_facts = [
        KBTriple(e1_label=f["e1_label"], relation=f["rel"], e2_label=f["e2_label"])
        for f in raw.get("kb_distractor_pool", [])
    ]

    return GraftSample(
        image_id=raw.get("image_id", "unknown"),
        scene_graph=scene_graph,
        oracle_facts=oracle_facts,
        distractor_facts=distractor_facts,
        query=raw.get("query", ""),
        ground_truth_trajectory=raw.get("ground_truth_trajectory", ""),
        answer=raw.get("answer", ""),
    )


class EntityMatchRetriever:
    """Simple KB retriever: returns triples where entity labels overlap with scene entities."""

    def __init__(self, knowledge_base: List[KBTriple]):
        self._kb = knowledge_base
        self._index = {}
        for triple in self._kb:
            for entity in [triple.e1_label, triple.e2_label]:
                if entity not in self._index:
                    self._index[entity] = []
                self._index[entity].append(triple)

    def retrieve(self, query: str, scene_entities: List[str], top_k: int = 5) -> List[KBTriple]:
        scored = {}
        for entity in scene_entities:
            for triple in self._index.get(entity, []):
                key = (triple.e1_label, triple.relation, triple.e2_label)
                scored[key] = scored.get(key, 0) + 1
            for kb_entity, triples in self._index.items():
                if entity in kb_entity or kb_entity in entity:
                    for triple in triples:
                        key = (triple.e1_label, triple.relation, triple.e2_label)
                        scored[key] = scored.get(key, 0) + 0.5

        sorted_keys = sorted(scored.keys(), key=lambda k: scored[k], reverse=True)[:top_k]
        return [KBTriple(e1_label=k[0], relation=k[1], e2_label=k[2]) for k in sorted_keys]


print("✅ Infrastructure defined")


In [ ]:
# Cell 4: Prompt Builder
import random

INSTRUCTION = "Isolate the correct reasoning path through the noisy graph context to answer the query."


def serialize_facts(facts: List[KBTriple], shuffle: bool = True) -> str:
    serialized = [f.serialize() for f in facts]
    if shuffle:
        random.shuffle(serialized)
    return "; ".join(serialized)


def build_training_prompt(sample: GraftSample, eos_token: str, shuffle_facts: bool = True) -> str:
    sg_context = sample.scene_graph.serialize_relations()
    facts_context = serialize_facts(sample.all_facts, shuffle=shuffle_facts)
    return (
        f"Instruction: {INSTRUCTION}\n"
        f"Scene Graph: {sg_context}\n"
        f"Graph-RAG Facts: {facts_context}\n"
        f"Query: {sample.query}\n"
        f"### Thought Traversal:\n{sample.ground_truth_trajectory}\n"
        f"### Answer:\n{sample.answer}{eos_token}"
    )


def build_inference_prompt(sample: GraftSample, shuffle_facts: bool = False) -> str:
    sg_context = sample.scene_graph.serialize_relations()
    facts_context = serialize_facts(sample.all_facts, shuffle=shuffle_facts)
    return (
        f"Instruction: {INSTRUCTION}\n"
        f"Scene Graph: {sg_context}\n"
        f"Graph-RAG Facts: {facts_context}\n"
        f"Query: {sample.query}\n"
        f"### Answer:\n"
    )


print("✅ Prompt builder defined")


In [ ]:
# Cell 5: Trainer
import os
import math
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from tqdm.auto import tqdm


class GraftDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=512, shuffle_facts=True):
        self.tokenizer = tokenizer
        self.max_length = max_length
        texts = [build_training_prompt(s, tokenizer.eos_token, shuffle_facts) for s in samples]

        self.encodings = tokenizer(
            texts, truncation=True, max_length=max_length, padding="max_length", return_tensors="pt"
        )

        labels = self.encodings["input_ids"].clone()
        answer_marker = "### Thought Traversal:\n"

        for i, text in enumerate(texts):
            marker_pos = text.find(answer_marker)
            if marker_pos == -1:
                continue
            prompt_portion = text[:marker_pos + len(answer_marker)]
            prompt_tokens = tokenizer(prompt_portion, truncation=True, max_length=max_length, return_tensors="pt")
            prompt_len = prompt_tokens["input_ids"].shape[1]
            attention_mask = self.encodings["attention_mask"][i]
            pad_len = (attention_mask == 0).sum().item()
            labels[i, :pad_len + prompt_len] = -100

        labels[self.encodings["attention_mask"] == 0] = -100
        self.encodings["labels"] = labels

    def __len__(self):
        return self.encodings["input_ids"].size(0)

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}


class GraftTrainer:
    def __init__(self, config):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model_name = config["model"]["name"]
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"
        self.model = AutoModelForCausalLM.from_pretrained(model_name).to(self.device)
        self.output_dir = config["output"]["dir"]
        os.makedirs(self.output_dir, exist_ok=True)

    def train(self, samples):
        cfg = self.config["training"]
        torch.manual_seed(cfg["seed"])

        dataset = GraftDataset(
            samples, self.tokenizer,
            max_length=self.config["model"]["max_length"],
            shuffle_facts=self.config["retrieval"]["shuffle_facts"],
        )

        total = len(dataset)
        train_len = int(total * cfg["train_split"])
        val_len = total - train_len
        train_ds, val_ds = random_split(dataset, [train_len, val_len])

        train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False)

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"])
        total_steps = len(train_loader) * cfg["epochs"]
        warmup_steps = int(total_steps * cfg["warmup_ratio"])
        scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

        best_val_loss = float("inf")
        all_metrics = []

        for epoch in range(1, cfg["epochs"] + 1):
            # Train
            self.model.train()
            total_loss = 0.0
            for batch in tqdm(train_loader, desc=f"Epoch {epoch} [train]"):
                batch = {k: v.to(self.device) for k, v in batch.items()}
                outputs = self.model(**batch)
                loss = outputs.loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                total_loss += loss.item()
            train_loss = total_loss / max(len(train_loader), 1)

            # Validate
            self.model.eval()
            val_total = 0.0
            with torch.no_grad():
                for batch in val_loader:
                    batch = {k: v.to(self.device) for k, v in batch.items()}
                    outputs = self.model(**batch)
                    val_total += outputs.loss.item()
            val_loss = val_total / max(len(val_loader), 1)

            perplexity = math.exp(val_loss) if val_loss < 100 else float("inf")
            metrics = TrainingMetrics(epoch=epoch, train_loss=train_loss, val_loss=val_loss, perplexity=perplexity)
            all_metrics.append(metrics)
            print(f"Epoch {epoch} — train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, ppl={perplexity:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                self._save("best")
                print(f"  → New best model saved")

        self._save("final")
        return all_metrics

    def _save(self, name):
        path = os.path.join(self.output_dir, name)
        os.makedirs(path, exist_ok=True)
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)


print("✅ Trainer defined")


In [ ]:
# Cell 6: Evaluator
import re
import numpy as np


def normalize_text(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def token_f1(pred: str, gold: str) -> float:
    p_tokens = pred.split()
    g_tokens = gold.split()
    if not p_tokens and not g_tokens:
        return 1.0
    if not p_tokens or not g_tokens:
        return 0.0
    common = {}
    for t in p_tokens:
        common[t] = common.get(t, 0) + 1
    match = 0
    for t in g_tokens:
        if common.get(t, 0) > 0:
            match += 1
            common[t] -= 1
    precision = match / len(p_tokens)
    recall = match / len(g_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def evaluate_model(model_path, samples, base_model_id="distilgpt2", max_new_tokens=64):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AutoModelForCausalLM.from_pretrained(model_path).to(device)
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(base_model_id)
    tokenizer.padding_side = "left"
    tokenizer.pad_token = tokenizer.eos_token

    results = []
    with torch.no_grad():
        for sample in samples:
            prompt = build_inference_prompt(sample, shuffle_facts=False)
            enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256, padding=True)
            enc = {k: v.to(device) for k, v in enc.items()}

            gen = model.generate(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                max_new_tokens=max_new_tokens,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )

            input_len = enc["input_ids"].shape[1]
            gen_tokens = gen[0, input_len:] if gen.shape[1] > input_len else gen[0]
            gen_answer = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

            gold = sample.answer.strip()
            norm_gen = normalize_text(gen_answer)
            norm_gold = normalize_text(gold)

            results.append(EvalResult(
                prompt=prompt, generated=gen_answer, gold=gold,
                exact_match=(norm_gen == norm_gold),
                token_f1=token_f1(norm_gen, norm_gold),
            ))

    exact_count = sum(1 for r in results if r.exact_match)
    avg_f1 = float(np.mean([r.token_f1 for r in results])) if results else 0.0
    return {"exact_match_rate": exact_count / len(results), "avg_token_f1": avg_f1, "results": results}


print("✅ Evaluator defined")


In [ ]:
# Cell 7: Sample Data
def get_diverse_dataset():
    samples = [
        {
            "image_id": "2407890",
            "gqa_scene_graph": {
                "objects": {"101": {"name": "delivery_van", "attributes": ["white", "electric"]}, "102": {"name": "charging_station", "attributes": ["active"]}},
                "relations": [{"source": "101", "name": "connected_to", "target": "102"}],
            },
            "fvqa_graph_rag_facts": [
                {"e1_label": "delivery_van", "rel": "IsA", "e2_label": "electric_vehicle"},
                {"e1_label": "charging_station", "rel": "ProvidesPowerTo", "e2_label": "electric_vehicle"},
                {"e1_label": "electric_vehicle", "rel": "Requires", "e2_label": "thermal_management"},
            ],
            "kb_distractor_pool": [
                {"e1_label": "street_light", "rel": "Emits", "e2_label": "yellow_light"},
                {"e1_label": "pedestrian", "rel": "WalksOn", "e2_label": "crosswalk"},
            ],
            "query": "What risk does this vehicle face if left unchanged over the next hour?",
            "ground_truth_trajectory": "Path: [Scene: delivery_van] -> connected_to -> [Scene: charging_station] <=> Aligned <=> <delivery_van, IsA, electric_vehicle> -> <charging_station, ProvidesPowerTo, electric_vehicle>.",
            "answer": "The delivery van is linked to an active charging station. Extended high-voltage power input risks battery degradation without active thermal management.",
        },
        {
            "image_id": "3501234",
            "gqa_scene_graph": {
                "objects": {"201": {"name": "stove", "attributes": ["gas", "lit"]}, "202": {"name": "towel", "attributes": ["cotton", "hanging"]}, "203": {"name": "pot", "attributes": ["metal"]}},
                "relations": [{"source": "202", "name": "near", "target": "201"}, {"source": "203", "name": "on", "target": "201"}],
            },
            "fvqa_graph_rag_facts": [
                {"e1_label": "cotton", "rel": "HasProperty", "e2_label": "flammable"},
                {"e1_label": "gas_stove", "rel": "Produces", "e2_label": "open_flame"},
                {"e1_label": "flammable_material", "rel": "IgnitesWhen", "e2_label": "near_open_flame"},
            ],
            "kb_distractor_pool": [
                {"e1_label": "pot", "rel": "MadeOf", "e2_label": "stainless_steel"},
                {"e1_label": "kitchen", "rel": "Contains", "e2_label": "refrigerator"},
            ],
            "query": "What safety hazard exists in this scene?",
            "ground_truth_trajectory": "Path: [Scene: towel] -> near -> [Scene: stove] <=> Aligned <=> <cotton, HasProperty, flammable> -> <gas_stove, Produces, open_flame>.",
            "answer": "The cotton towel hanging near the lit gas stove is a fire hazard. Cotton is flammable and proximity to an open flame creates ignition risk.",
        },
        {
            "image_id": "4602345",
            "gqa_scene_graph": {
                "objects": {"301": {"name": "bicycle", "attributes": ["moving"]}, "302": {"name": "truck", "attributes": ["large", "turning"]}, "303": {"name": "traffic_light", "attributes": ["green"]}},
                "relations": [{"source": "301", "name": "beside", "target": "302"}, {"source": "302", "name": "approaching", "target": "303"}],
            },
            "fvqa_graph_rag_facts": [
                {"e1_label": "truck", "rel": "HasProperty", "e2_label": "large_blind_spot"},
                {"e1_label": "bicycle", "rel": "IsA", "e2_label": "vulnerable_road_user"},
                {"e1_label": "blind_spot", "rel": "Causes", "e2_label": "collision_risk"},
            ],
            "kb_distractor_pool": [
                {"e1_label": "traffic_light", "rel": "Controls", "e2_label": "intersection_flow"},
                {"e1_label": "road", "rel": "HasMarking", "e2_label": "lane_divider"},
            ],
            "query": "What danger does the cyclist face?",
            "ground_truth_trajectory": "Path: [Scene: bicycle] -> beside -> [Scene: truck] <=> Aligned <=> <truck, HasProperty, large_blind_spot> -> <blind_spot, Causes, collision_risk>.",
            "answer": "The cyclist is beside a turning truck. Trucks have large blind spots, and the cyclist as a vulnerable road user faces collision risk if the truck turns without seeing them.",
        },
        {
            "image_id": "5703456",
            "gqa_scene_graph": {
                "objects": {"401": {"name": "patient", "attributes": ["elderly"]}, "402": {"name": "medication_bottle", "attributes": ["open", "multiple"]}, "403": {"name": "glass", "attributes": ["empty"]}},
                "relations": [{"source": "401", "name": "holding", "target": "402"}, {"source": "403", "name": "on_table_near", "target": "401"}],
            },
            "fvqa_graph_rag_facts": [
                {"e1_label": "elderly_patient", "rel": "AtRiskOf", "e2_label": "polypharmacy"},
                {"e1_label": "multiple_medications", "rel": "Causes", "e2_label": "drug_interaction"},
                {"e1_label": "drug_interaction", "rel": "LeadsTo", "e2_label": "adverse_effects"},
            ],
            "kb_distractor_pool": [
                {"e1_label": "glass", "rel": "MadeOf", "e2_label": "transparent_material"},
                {"e1_label": "chair", "rel": "UsedFor", "e2_label": "sitting"},
            ],
            "query": "What medical concern is suggested by this scene?",
            "ground_truth_trajectory": "Path: [Scene: patient] -> holding -> [Scene: medication_bottle(multiple)] <=> Aligned <=> <elderly_patient, AtRiskOf, polypharmacy> -> <multiple_medications, Causes, drug_interaction>.",
            "answer": "The elderly patient with multiple open medication bottles suggests polypharmacy risk. Multiple medications increase the chance of drug interactions and adverse effects.",
        },
        {
            "image_id": "6804567",
            "gqa_scene_graph": {
                "objects": {"501": {"name": "worker", "attributes": ["no_helmet"]}, "502": {"name": "crane", "attributes": ["operating"]}, "503": {"name": "steel_beam", "attributes": ["suspended"]}},
                "relations": [{"source": "502", "name": "lifting", "target": "503"}, {"source": "501", "name": "below", "target": "503"}],
            },
            "fvqa_graph_rag_facts": [
                {"e1_label": "suspended_load", "rel": "HasRisk", "e2_label": "falling_object"},
                {"e1_label": "hard_hat", "rel": "Protects", "e2_label": "head_injury"},
                {"e1_label": "worker_without_ppe", "rel": "Violates", "e2_label": "safety_regulation"},
            ],
            "kb_distractor_pool": [
                {"e1_label": "crane", "rel": "OperatedBy", "e2_label": "certified_operator"},
                {"e1_label": "construction_site", "rel": "Requires", "e2_label": "permit"},
            ],
            "query": "What safety violation is occurring?",
            "ground_truth_trajectory": "Path: [Scene: worker(no_helmet)] -> below -> [Scene: steel_beam(suspended)] <=> Aligned <=> <suspended_load, HasRisk, falling_object> -> <worker_without_ppe, Violates, safety_regulation>.",
            "answer": "A worker without a helmet is positioned below a suspended steel beam. This violates safety regulations as falling objects from overhead crane operations require head protection.",
        },
    ]
    # Scale to 40 samples
    return samples * 8


raw_data = get_diverse_dataset()
samples = [parse_raw_sample(r) for r in raw_data]
print(f"✅ Loaded {len(samples)} training samples")


In [ ]:
# Cell 8: Train the Model
config = {
    "model": {"name": "distilgpt2", "max_length": 512},
    "training": {
        "epochs": 3, "batch_size": 2, "learning_rate": 5e-5,
        "weight_decay": 0.01, "warmup_ratio": 0.1, "train_split": 0.8, "seed": 42,
    },
    "output": {"dir": "./graft_model", "save_best_only": True},
    "retrieval": {"shuffle_facts": True},
}

trainer = GraftTrainer(config)
metrics = trainer.train(samples)

print("\n=== Training Summary ===")
for m in metrics:
    print(f"  Epoch {m.epoch}: train={m.train_loss:.4f}, val={m.val_loss:.4f}, ppl={m.perplexity:.4f}")


In [ ]:
# Cell 9: Evaluate
val_samples = samples[int(len(samples) * 0.8):]
metrics = evaluate_model("./graft_model/best", val_samples)

print(f"Exact Match Rate: {metrics['exact_match_rate']:.2%}")
print(f"Avg Token-F1: {metrics['avg_token_f1']:.4f}")

print("\nSample predictions:")
for r in metrics["results"][:3]:
    print(f"  Gold: {r.gold[:80]}")
    print(f"  Gen:  {r.generated[:80]}")
    print(f"  F1:   {r.token_f1:.4f}\n")


In [ ]:
# Cell 10: Custom Inference
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForCausalLM.from_pretrained("./graft_model/best").to(device)
model.eval()
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Build a custom scene
scene = SceneGraph(
    objects={"1": SceneObject("1", "car", ["red", "parked"]), "2": SceneObject("2", "tree", ["large"])},
    relations=[SceneRelation("1", "under", "2")],
)
test_sample = GraftSample(
    image_id="custom",
    scene_graph=scene,
    oracle_facts=[KBTriple("tree", "CanDrop", "branches"), KBTriple("falling_branches", "Damage", "vehicle")],
    distractor_facts=[KBTriple("car", "HasColor", "red")],
    query="What risk does the car face?",
)

prompt = build_inference_prompt(test_sample)
enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256, padding=True)
enc = {k: v.to(device) for k, v in enc.items()}

with torch.no_grad():
    gen = model.generate(
        input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
        max_new_tokens=64, do_sample=True, top_k=50, top_p=0.95,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
    )

input_len = enc["input_ids"].shape[1]
answer = tokenizer.decode(gen[0, input_len:], skip_special_tokens=True).strip()
print(f"Query: {test_sample.query}")
print(f"Answer: {answer}")